# DantinoX — EMNLP 2026 System Demo

[![GitHub](https://img.shields.io/badge/GitHub-DantinoX-blue)](https://github.com/winstonsmith1897/DantinoX)
[![License: MIT](https://img.shields.io/badge/License-MIT-green.svg)](https://opensource.org/licenses/MIT)

**DantinoX** is a JAX/Flax NNX library for training and comparing language models across
**three generation paradigms** (AR, Discrete Diffusion, ELF flow-matching) on the
**same transformer backbone**, with the same trainer and the same benchmark suite.

This notebook is the official demo for the EMNLP 2026 System Demonstration track.

| Section | Content | Est. time |
|---|---|---|
| 1 — Setup | install + version check | ~15 s |
| 2 — ModelConfig | backbone, attention, FFN, norm, positional encoding | instant |
| 3 — FLOPs profile | analytical FLOPs without a GPU | instant |
| 4 — Three paradigms | AR / Discrete Diffusion / ELF | instant |
| 5 — Parallelism | Data Parallel + Tensor Parallel | instant |
| 6 — Live training | tiny model (dim=128, 4 blocks, 3 epochs) | ~30 s |
| 7 — Generation | tiny trained model + pre-trained checkpoints | ~10 s |
| 8 — Full Trainer | complete paradigm + trainer assembly | ~5 s |
| 9 — Results | 279-experiment benchmark, figures, training curves | instant |

> **Recommended runtime**: single GPU — A100 / RTX 3090 or better.
> All configuration cells are CPU-only and do not require a GPU.

---
## 1 — Setup

Install DantinoX in **editable mode** (`-e`) from the local source tree.
With `-e`, Python reads directly from the repo — no reinstall needed after code changes.

In [ ]:
import subprocess
import sys

REPO = "/ssd1/marco.simoni/VULNERABILITY/NETGROUP/DantinoX"
subprocess.check_call([sys.executable, "-m", "pip", "install", "-e", REPO, "-q"])
print("\u2705 DantinoX installed")

In [ ]:
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "6"   # change to your available GPU

import jax
from flax import nnx

import dantinox as dx
from dantinox import count_flops

dx.banner(dx.__version__)

print(f"dantinox  {dx.__version__}")
print(f"JAX       {jax.__version__}")
print(f"Devices:  {jax.devices()}")


def param_count(cfg) -> int:
    """Count trainable parameters without running the model."""
    m = dx.Paradigm(cfg).build_model(nnx.Rngs(0))
    return sum(x.size for x in jax.tree_util.tree_leaves(nnx.state(m, nnx.Param)))

---
## 2 — ModelConfig: the entire architecture in one object

`ModelConfig` is the **single control point** for the model architecture.
Changing one field produces a completely different variant — no YAML config files,
no argparse, pure Python.

### Key parameters

| Parameter | Default | Description |
|---|---|---|
| `paradigm` | `"ar"` | Generation paradigm: `"ar"`, `"discrete"`, `"continuous"` |
| `dim` | `256` | Model width (hidden dimension) |
| `n_heads` | `4` | Number of attention heads |
| `num_blocks` | `4` | Number of transformer blocks |
| `vocab_size` | `32128` | Vocabulary size |
| `attention` | `"mha"` | Attention mechanism: `"mha"`, `"gqa"`, `"mla"` |
| `ffn` | `"mlp"` | FFN type: `"mlp"` or `"moe"` |
| `use_swiglu` | `True` | SwiGLU gate in the MLP |
| `norm` | `"rmsnorm"` | Normalization: `"rmsnorm"`, `"layernorm"` |
| `pos_encoding` | `"rotary"` | Positional encoding: `"rotary"`, `"learned"`, `"absolute"`, `"none"` |
| `tp_size` | `1` | Tensor parallel shards |

---
### 2a — Attention variants

| `attention=` | Name | KV cache size | Typical use |
|---|---|---|---|
| `"mha"` | Multi-Head Attention | `n_heads × seq × head_dim` | classic baseline |
| `"gqa"` | Grouped-Query Attention | `kv_heads × seq × head_dim` | Llama 3, Mistral |
| `"mla"` | Multi-Latent Attention | `down_dim_kv × seq` (latent!) | DeepSeek-V2/V3 |
| `"mha"` + `sliding_window=True` | Sliding-Window | only `context_window` tokens | Mistral, Phi-3 |

**GQA** reduces the KV cache by a factor of `n_heads / kv_heads`.
With `kv_heads=2` and `n_heads=8`, the KV cache is **4× smaller** than MHA.

**MLA** compresses further: instead of storing K and V separately, it saves a
**single compressed latent vector** and reconstructs K/V on-the-fly.
The cache savings depend on `down_dim_kv` (default: `dim // 4`).

**SWA** restricts attention to a `context_window`-token window — O(n) instead of O(n²).

In [ ]:
# Shared backbone parameters — only attention= changes
COMMON = dict(
    paradigm   = "ar",
    ffn        = "mlp",
    use_swiglu = True,
    dim        = 512,
    n_heads    = 8,
    num_blocks = 12,
    vocab_size = 32128,
)

attn_variants = [
    ("MHA  — Multi-Head Attention (baseline)",
     dx.ModelConfig(attention="mha", **COMMON)),
    ("GQA  — Grouped-Query  (kv_heads=2, 4x smaller KV cache)",
     dx.ModelConfig(attention="gqa", kv_heads=2, **COMMON)),
    ("GQA  — Grouped-Query  (kv_heads=1 = Multi-Query)",
     dx.ModelConfig(attention="gqa", kv_heads=1, **COMMON)),
    ("MLA  — Multi-Latent Attention (DeepSeek-style)",
     dx.ModelConfig(attention="mla", **COMMON)),
    ("SWA  — Sliding-Window  (context_window=128)",
     dx.ModelConfig(attention="mha", sliding_window=True, context_window=128, **COMMON)),
]

print(f"{'Variant':<50}  {'Params':>10}")
print("-" * 64)
for name, cfg in attn_variants:
    n = param_count(cfg)
    print(f"  {name:<48}  {n/1e6:>8.1f}M")

print()
# Detailed repr for the main variants
for name, cfg in attn_variants[:4]:
    print(f"\n{'\u2500'*60}")
    print(f"  {name}")
    print(f"{'\u2500'*60}")
    print(cfg)

### 2b — FFN variants

| `ffn=` | `use_swiglu` | Activation | Extra params | Notes |
|---|---|---|---|---|
| `"mlp"` | `False` | GELU | — | classic Transformer |
| `"mlp"` | `True` | SwiGLU | — | default · PaLM, Llama 2/3 |
| `"moe"` | `True` | SwiGLU × expert | `n_experts`, `top_k` | sparse · sub-linear FLOPs |
| `"moe"` | `True` | SwiGLU latent | `moe_latent=True`, `moe_latent_dim` | experts in compressed space |

**SwiGLU** splits the linear output into two halves: `output = value × σ(gate)`.
The gate is learned and per-token, making it more expressive than GELU at the same FLOPs
(the hidden dimension is scaled to `8/3·dim` to compensate for the split).

**MoE (Mixture of Experts)** replaces the dense MLP with `n_experts` parallel networks.
A learned router selects `top_k` experts per token — only those activate.
Result: **parameters grow** linearly with `n_experts`, but **FLOPs grow** only with `top_k`.

**MoE-Latent**: experts operate in a compressed space (`moe_latent_dim`).
Often paired with MLA to maximise model compression.

In [ ]:
COMMON_GQA = dict(
    paradigm   = "ar",
    attention  = "gqa",
    kv_heads   = 2,
    dim        = 512,
    n_heads    = 8,
    num_blocks = 12,
    vocab_size = 32128,
)

ffn_variants = [
    ("MLP + GELU  (classic, use_swiglu=False)",
     dx.ModelConfig(ffn="mlp", use_swiglu=False, activation="gelu", **COMMON_GQA)),
    ("MLP + SwiGLU  (default — Llama/PaLM style)",
     dx.ModelConfig(ffn="mlp", use_swiglu=True, **COMMON_GQA)),
    ("MoE  (8 experts, top-2 per token)",
     dx.ModelConfig(ffn="moe", n_experts=8, top_k=2, **COMMON_GQA)),
    ("MoE-Latent  (experts in 64-d, moe_latent=True)",
     dx.ModelConfig(ffn="moe", n_experts=8, top_k=2,
                    moe_latent=True, moe_latent_dim=64,
                    attention="mla",
                    **dict(paradigm="ar", dim=512, n_heads=8,
                           num_blocks=12, vocab_size=32128))),
]

print(f"{'FFN variant':<50}  {'Params (512d/12b)':>18}")
print("-" * 72)
for name, cfg in ffn_variants:
    n = param_count(cfg)
    print(f"  {name:<48}  {n/1e6:>16.1f}M")

print()
for name, cfg in ffn_variants:
    print(f"\n{'\u2500'*60}")
    print(f"  {name}")
    print(f"{'\u2500'*60}")
    print(cfg)

### 2c — Norm & Positional Encoding

| `norm=` | `pos_encoding=` | Notes |
|---|---|---|
| `"rmsnorm"` | `"rotary"` | **Default** — RoPE is relative, no positional embedding matrix |
| `"layernorm"` | `"learned"` | Classic BERT-style learned absolute positions |
| `"rmsnorm"` | `"absolute"` | Fixed sinusoidal (Vaswani et al. 2017) |
| `"rmsnorm"` | `"none"` | No position info — useful for sequences with implicit order |

**RMSNorm** is simpler and slightly faster than LayerNorm:
it normalises only by the root mean square (no bias, no mean subtraction).

**RoPE** (Rotary Positional Embedding) encodes position by rotating the Q/K space:
relative distance is naturally captured by the dot product.

In [ ]:
norm_pos_variants = [
    ("RMSNorm + RoPE       (default)",
     dx.ModelConfig(norm="rmsnorm",  pos_encoding="rotary",   **COMMON_GQA)),
    ("LayerNorm + Learned  (BERT-style)",
     dx.ModelConfig(norm="layernorm", pos_encoding="learned",  **COMMON_GQA)),
    ("RMSNorm + Sinusoidal",
     dx.ModelConfig(norm="rmsnorm",  pos_encoding="absolute",  **COMMON_GQA)),
    ("RMSNorm + None       (no position info)",
     dx.ModelConfig(norm="rmsnorm",  pos_encoding="none",      **COMMON_GQA)),
]

print(f"{'Norm / Pos variant':<44}  {'Params':>10}  Note")
print("-" * 72)
for name, cfg in norm_pos_variants:
    n = param_count(cfg)
    delta = "(+pos embed matrix)" if cfg.pos_encoding == "learned" else ""
    print(f"  {name:<42}  {n/1e6:>8.1f}M  {delta}")

---
## 3 — Analytical FLOPs profile (no GPU required)

`count_flops(cfg, seq_len)` computes FLOPs **analytically** — no GPU computation,
result in milliseconds. Useful for comparing architectures *before* launching training.

The breakdown distinguishes:
- **attention**: `Q·Kᵀ` (quadratic in `seq_len`) + softmax + `·V` + out-proj
- **ffn**: the two linear layers of the MLP or the **active** experts in MoE (`top_k / n_experts` × dense)
- **embedding**: initial lookup + final logit projection over vocab

For MoE, FFN FLOPs are `top_k / n_experts` of the dense case,
but total parameters are `n_experts` times larger.

In [ ]:
flops_configs = [
    ("MHA  + SwiGLU   (dim=128, 4b)",
     dx.ModelConfig(attention="mha", ffn="mlp", use_swiglu=True,
                    dim=128, n_heads=4, num_blocks=4, vocab_size=32128, paradigm="ar")),
    ("GQA  + SwiGLU   (dim=512, 12b)",
     dx.ModelConfig(attention="gqa", kv_heads=2, ffn="mlp", use_swiglu=True,
                    dim=512, n_heads=8, num_blocks=12, vocab_size=32128, paradigm="ar")),
    ("GQA  + MoE 8exp (dim=512, 12b)",
     dx.ModelConfig(attention="gqa", kv_heads=2, ffn="moe", n_experts=8, top_k=2,
                    dim=512, n_heads=8, num_blocks=12, vocab_size=32128, paradigm="ar")),
    ("MLA  + MoE-Lat  (dim=512, 12b)",
     dx.ModelConfig(attention="mla", ffn="moe", n_experts=8, top_k=2,
                    moe_latent=True, moe_latent_dim=64,
                    dim=512, n_heads=8, num_blocks=12, vocab_size=32128, paradigm="ar")),
    ("GQA  + SwiGLU   (dim=768, 16b)",
     dx.ModelConfig(attention="gqa", kv_heads=2, ffn="mlp", use_swiglu=True,
                    dim=768, n_heads=12, num_blocks=16, vocab_size=32128, paradigm="ar")),
]

SEQ = 256
print(f"FLOPs @ seq_len={SEQ}, batch=1\n")
print(f"{'Config':<40}  {'Params':>8}  Breakdown")
print("=" * 70)

for name, cfg in flops_configs:
    n  = param_count(cfg)
    fl = count_flops(cfg, seq_len=SEQ)
    print(f"\n{name}  ({n/1e6:.1f}M params)")
    print(fl)

---
## 4 — Three paradigms, one backbone

The `paradigm=` field in `ModelConfig` selects the **generation strategy**.
Everything else — architecture, optimiser, training loop, CLI — stays **identical**.

| `paradigm=` | Name | Model input | Loss | Inference |
|---|---|---|---|---|
| `"ar"` | Autoregressive | previous tokens (causal mask) | cross-entropy | greedy / top-k / nucleus |
| `"discrete"` | Discrete Diffusion | partially masked tokens | cross-entropy on `[MASK]` | iterative denoising (T → 0) |
| `"continuous"` | ELF flow-matching | embeddings + Gaussian noise | MSE on trajectory | ODE solver (N steps) |

**AR**: at each step the model sees previous tokens and predicts the next one.
Trained with a causal mask. Sequential token-by-token generation.

**Discrete Diffusion (LLaDA)**: at timestep `t=T` all tokens are `[MASK]`.
At each step the model sees the full bidirectional context and unmasks a fraction of tokens.
At `t=0` the sequence is fully generated.
Non-autoregressive: all tokens are generated **in parallel** at each step.

**ELF (Embedded Language Flows)**: maps Gaussian noise to token embeddings via a continuous ODE.
A frozen T5 encoder provides prompt conditioning.
Fully non-autoregressive — no causal dependencies.

In [ ]:
# Same backbone — only paradigm= changes
BACKBONE = dict(
    attention  = "gqa",
    kv_heads   = 2,
    ffn        = "mlp",
    use_swiglu = True,
    dim        = 512,
    n_heads    = 8,
    num_blocks = 12,
    vocab_size = 32128,
)

paradigm_variants = [
    ("AR  — causal next-token prediction (GPT-style)", "ar"),
    ("Discrete Diffusion  — LLaDA iterative unmasking", "discrete"),
    ("ELF  — continuous embedding flow-matching", "continuous"),
]

for name, p in paradigm_variants:
    cfg = dx.ModelConfig(paradigm=p, **BACKBONE)
    print("\n" + "\u2550" * 60)
    print(f"  {name}")
    print("\u2550" * 60)
    print(cfg)
    # Paradigm wraps the model with the paradigm-specific loss and sampler
    print(dx.Paradigm(cfg))

---
## 5 — Distributed parallelism: DP and TP

DantinoX supports two **composable** parallelism axes:

### Data Parallel (DP)
Each GPU holds a **complete** copy of the model and processes different micro-batches.
Gradients are averaged with `all-reduce` at the end of every backward pass.
- Config: `n_devices=N` in `TrainingConfig`, `tp_size=1` in both configs
- Overhead: gradient communication only — O(parameters)
- Best when: the model fits on a single GPU

### Tensor Parallel (TP)
Weight matrices are split along the hidden dimension across `tp_size` GPUs.
Each GPU computes one shard; activations are synchronised with `all-gather` / `reduce-scatter`.
- Config: `tp_size=T` in **both** `ModelConfig` and `TrainingConfig`
- Overhead: activation communication — O(batch × seq × dim)
- Best when: the model does not fit on a single GPU

### Combined DP × TP
Both can be used simultaneously:
`n_devices=4, tp_size=2` → 2 DP replicas × 2 TP shards = **4 GPUs total**.

| Scheme | `tp_size` | `n_devices` | Total GPUs | Overhead |
|---|---|---|---|---|
| DP only | 1 | N | N | gradients |
| TP only | T | 1 | T | activations |
| DP × TP | T | R | R × T | both |

In [ ]:
parallel_variants = [
    (
        "Pure Data Parallel  (4 replicas, full model on each GPU)",
        dict(paradigm="ar", attention="gqa", kv_heads=2, ffn="mlp",
             tp_size=1, dim=512, n_heads=8, num_blocks=12, vocab_size=32128),
        dict(n_devices=4, tp_size=1, lr=3e-4, batch_size=64),
    ),
    (
        "Pure Tensor Parallel  (1 replica, model sharded across 4 GPUs)",
        dict(paradigm="ar", attention="gqa", kv_heads=2, ffn="mlp",
             tp_size=4, dim=512, n_heads=8, num_blocks=12, vocab_size=32128),
        dict(n_devices=1, tp_size=4, lr=3e-4, batch_size=64),
    ),
    (
        "DP \u00d7 TP combined  (2 replicas \u00d7 2 TP shards = 4 GPUs)",
        dict(paradigm="discrete", attention="mla", ffn="moe",
             n_experts=8, top_k=2, tp_size=2,
             dim=768, n_heads=12, num_blocks=16, vocab_size=32128),
        dict(n_devices=2, tp_size=2, lr=3e-4, batch_size=64, grad_accum=4),
    ),
]

for name, mkw, tkw in parallel_variants:
    mc = dx.ModelConfig(**mkw)
    tc = dx.TrainingConfig(**tkw)
    print("\n" + "\u2550" * 60)
    print(f"  {name}")
    print("\u2550" * 60)
    print(mc)
    print(tc)

---
## 6 — Live training (~30 seconds)

Real-time training on **Tiny Shakespeare** with a small but real model.

Why tiny? For the video demo we want to see the loss drop *live*:
tqdm shows step, time-per-step, and loss at every epoch.
The pre-trained checkpoints in section 7 show full-scale results.

### Tiny architecture

| Parameter | Value | Rationale |
|---|---|---|
| `dim` | `128` | vs 768 of production runs — 36× fewer parameters |
| `n_heads` | `4` | proportional to dim |
| `num_blocks` | `4` | vs 16 of production runs |
| `tokenizer_type` | `"char"` | vocab=256, zero pre-processing, faster than BPE |
| `epochs` | `3` | ~10 steps/epoch on this GPU → ~30 s total |

The `tiny_shakespeare.txt` corpus (~1 MB, already in the repo) contains
the complete works of Shakespeare — enough to see learning in a few epochs.

In [ ]:
CORPUS = "docs/notebooks/tiny_shakespeare.txt"   # ~1 MB, already in the repo

# ── Tiny model: trains in ~30 s on one GPU ──────────────────────────────────
tiny_cfg = dx.ModelConfig(
    paradigm   = "ar",
    attention  = "gqa",   kv_heads   = 2,
    ffn        = "mlp",   use_swiglu = True,
    dim        = 128,
    n_heads    = 4,
    num_blocks = 4,
    vocab_size = 256,     # char tokenizer: 256 distinct ASCII characters
)

tiny_train = dx.TrainingConfig(
    lr             = 3e-4,
    epochs         = 3,
    batch_size     = 64,
    tokenizer_type = "char",
    val_frac       = 0.1,
    eval_iters     = 20,
)

n_params = param_count(tiny_cfg)
print(f"Tiny model: {n_params/1e3:.0f}K parameters")
print(tiny_cfg)
print("\n" + tiny_train.__repr__())
print("\n" + "\u2500" * 50)
print("Training — watch the loss drop:")
print("\u2500" * 50)

tiny_run = dx.Trainer(dx.Paradigm(tiny_cfg), tiny_train).fit(CORPUS)
print(f"\n\u2705 Checkpoint saved to: {tiny_run}")

---
## 7 — Text generation

`Generator` wraps any trained checkpoint with multiple decoding strategies.
The tokenizer is loaded automatically from the checkpoint — no manual configuration needed.

### Decoding strategies

| Strategy | Parameters | When to use |
|---|---|---|
| Greedy | `greedy=True` | deterministic, maximum-probability output |
| Top-k | `top_k=K, temperature=T` | controlled diversity — k=40, T=0.8 is a good default |
| Nucleus | `top_p=P, temperature=T` | selects the minimum nucleus with prob ≥ P |
| Streaming | `gen.stream(...)` | prints token-by-token, great for video demos |

### 7a — Tiny trained model (just learned)

In [ ]:
from dantinox import Generator

gen_tiny = Generator(tiny_run, seed=42)
PROMPT = "HAMLET:\n"

print(f"Prompt: {repr(PROMPT)}")
print(f"Model:  tiny GQA-AR ({param_count(tiny_cfg)/1e3:.0f}K params, 3 epochs)\n")

print("=" * 55)
print("  Greedy (deterministic, T=0.0)")
print("=" * 55)
print(gen_tiny.generate(PROMPT, max_new_tokens=200, greedy=True))

print("\n" + "=" * 55)
print("  Top-k  (k=40, temperature=0.8)")
print("=" * 55)
print(gen_tiny.generate(PROMPT, max_new_tokens=200, top_k=40, temperature=0.8))

print("\n" + "=" * 55)
print("  Nucleus  (top_p=0.9, temperature=0.9)")
print("=" * 55)
print(gen_tiny.generate(PROMPT, max_new_tokens=200, top_p=0.9, temperature=0.9))

### 7b — Pre-trained checkpoints (768d · 16 blocks · WikiText-103)

Same `Generator` API — only `run_dir` changes.
The models below were trained as part of the 279-experiment benchmark suite.

| Checkpoint | Paradigm | Attention | FFN | Params |
|---|---|---|---|---|
| `diff_mha_768d_16b_Dense` | Discrete Diffusion | MHA | SwiGLU | ~235M |
| `diff_gqa_768d_16b_MoE8exp` | Discrete Diffusion | GQA | MoE 8exp | ~890M |
| `elf_mha_768d_16b_Dense` | ELF flow-matching | MHA | SwiGLU | ~235M |
| `elf_mla_768d_16b_Dense` | ELF flow-matching | MLA | SwiGLU | ~225M |

In [ ]:
PROMPT_EN = "Language models will change"

checkpoints = [
    ("Discrete Diffusion \u00b7 MHA \u00b7 Dense",   "runs/diff_mha_768d_16b_Dense"),
    ("Discrete Diffusion \u00b7 GQA \u00b7 MoE 8exp", "runs/diff_gqa_768d_16b_MoE8exp"),
    ("ELF \u00b7 MHA \u00b7 Dense",                   "runs/elf_mha_768d_16b_Dense"),
    ("ELF \u00b7 MLA \u00b7 Dense",                   "runs/elf_mla_768d_16b_Dense"),
]

for name, run_dir in checkpoints:
    if not os.path.isdir(run_dir):
        print(f"[skip] {name} — {run_dir} not found")
        continue
    print("\n" + "\u2550" * 60)
    print(f"  {name}")
    print("\u2550" * 60)
    gen = Generator(run_dir, seed=42)
    print(gen.generate(PROMPT_EN, max_new_tokens=120, top_k=50, temperature=0.8))

### 7c — Token-by-token streaming (discrete diffusion)

With `Generator.stream()` chunks are printed as they are generated.

For **discrete diffusion** this is particularly visual:
you can see the sequence "clear up" iteratively as the model removes masks.
Each step denoises another fraction of the masked tokens, converging to the final sequence.

In [ ]:
DIFF_RUN = "runs/diff_mha_768d_16b_Dense"

if os.path.isdir(DIFF_RUN):
    gen_diff = Generator(DIFF_RUN, seed=42)
    print("Streaming discrete diffusion (50 denoising steps):")
    print(f"Prompt: {repr(PROMPT_EN)}")
    print("\u2500" * 50)
    for chunk in gen_diff.stream(PROMPT_EN, n_steps=50, max_new_tokens=100):
        print(chunk, end="", flush=True)
    print()
else:
    print(f"[skip] {DIFF_RUN} not found — falling back to tiny model")
    for chunk in gen_tiny.stream(PROMPT, max_new_tokens=100, top_k=40, temperature=0.8):
        print(chunk, end="", flush=True)
    print()

---
## 8 — Full Trainer: complete assembly

The Trainer assembles three pieces into a single pipeline:

```
ModelConfig  ──►  Paradigm  ──┐
                              ├──►  Trainer.fit()  ──►  run_dir/
TrainingConfig  ─────────────┘
```

The output `run_dir/` contains:
- `best_model_weights.msgpack` — checkpoint with best val_loss
- `model_weights.msgpack` — final checkpoint
- `config.yaml` — serialised ModelConfig + TrainingConfig
- `tokenizer.json` — tokenizer used in training
- `model_summary.json` — total params, weight/optimiser memory
- `training_log.csv` — step, train_loss, val_loss, ms_per_step

### TrainingConfig — key parameters

| Parameter | Default | Description |
|---|---|---|
| `lr` | `3e-4` | Initial learning rate |
| `epochs` | `10` | Number of epochs |
| `batch_size` | `64` | Mini-batch per GPU |
| `grad_accum` | `1` | Gradient accumulation steps |
| `optimizer` | `"adamw"` | Optimiser: `"adamw"`, `"sgd"`, `"lion"` |
| `lr_schedule` | `"cosine"` | Schedule: `"cosine"`, `"constant"`, `"linear"` |
| `warmup_steps` | `400` | Linear LR warmup steps |
| `n_devices` | `1` | Number of DP replicas |
| `tp_size` | `1` | Tensor parallel shards (must match ModelConfig) |
| `dataset_name` | — | HuggingFace dataset name or local path |
| `tokenizer_type` | `"t5"` | `"t5"`, `"bpe"`, `"char"` |
| `val_frac` | `0.1` | Validation split fraction |
| `eval_iters` | `50` | Validation steps per checkpoint |

In [ ]:
# ── ModelConfig: Discrete Diffusion + GQA + MoE + SwiGLU + 2-way TP ──────
cfg = dx.ModelConfig(
    paradigm   = "discrete",  # LLaDA-style masked diffusion
    attention  = "gqa",       # Grouped-Query: smaller KV cache
    kv_heads   = 2,
    ffn        = "moe",       # Mixture of Experts
    n_experts  = 8,
    top_k      = 2,           # 2 active experts per token
    use_swiglu = True,        # SwiGLU gate (default)
    tp_size    = 2,           # 2-way tensor parallel
    dim        = 768,
    n_heads    = 12,
    num_blocks = 16,
    vocab_size = 32128,
)

# ── TrainingConfig: AdamW + cosine schedule + 4 GPUs (2 DP × 2 TP) ────────
tcfg = dx.TrainingConfig(
    lr             = 3e-4,
    epochs         = 10,
    batch_size     = 64,
    grad_accum     = 4,          # effective batch = 64 × 4 = 256
    optimizer      = "adamw",
    lr_schedule    = "cosine",
    warmup_steps   = 400,
    n_devices      = 4,          # 2 DP replicas × 2 TP shards = 4 GPUs
    tp_size        = 2,
    dataset_name   = "wikitext",
    dataset_source = "huggingface",
    dataset_config = "wikitext-103-raw-v1",
    tokenizer_type = "t5",
)

# ── Paradigm: wraps model + paradigm-specific loss + sampler ─────────────
paradigm = dx.Paradigm(cfg)

print("ModelConfig — Discrete Diffusion, GQA+MoE, 2-way TP:")
print(cfg)
print("\nTrainingConfig — AdamW cosine, 4 GPUs (2DP×2TP):")
print(tcfg)
print("\nParadigm:")
print(paradigm)

# ── Trainer: one call to launch training ─────────────────────────────────
trainer = dx.Trainer(paradigm, tcfg)
print("\n\u2705 Trainer ready.")
print("   To launch training on WikiText-103 (4 GPUs):")
print("   >>> run_dir = trainer.fit()")

# Uncomment to start (requires 4 GPUs + HuggingFace dataset, takes hours)
# run_dir = trainer.fit()

---
## 9 — Results: 279 experiments

The benchmark suite was run on **8× A100-PCIE-40GB** covering all combinations of:

| Dimension | Paradigm | Attention | FFN | Total |
|---|---|---|---|---|
| 512d/12b, 768d/16b | AR, Discrete, ELF | MHA, GQA, MLA | Dense, MoE 8exp | **279 runs** |

**Training corpus**: WikiText-103 (~100M tokens) with T5 tokenizer (32128 tokens).

**Collected metrics**:

| Metric | What it measures |
|---|---|
| throughput (tok/s) | generated tokens per second (inference) |
| latency (ms/tok) | ms per generated token (first-token latency) |
| MFU | Model FLOP Utilisation vs peak GPU FLOP/s |
| MAUVE | quality of the generated distribution (vs reference) |
| distinct-2 | lexical diversity of generated texts |
| repetition rate | n-gram repetition rate |
| bpb | bits-per-byte (perplexity proxy) |

In [ ]:
import matplotlib.image as mpimg
import matplotlib.pyplot as plt

FIG_DIR = "results/emnlp_figs"

figures = [
    ("fig_emnlp_1_pareto.png",
     "Fig 1 — Pareto: throughput (tok/s) vs generation quality (MAUVE)\n"
     "ELF dominates at high quality; Diffusion+MoE optimal for throughput"),
    ("fig_emnlp_2_quality.png",
     "Fig 2 — Quality metrics by paradigm\n"
     "MAUVE, distinct-2, repetition rate, bpb"),
    ("fig_emnlp_3_attention.png",
     "Fig 3 — Attention variants: MHA vs GQA vs MLA\n"
     "Quality, throughput, and KV-cache footprint"),
    ("fig_emnlp_4_scaling.png",
     "Fig 4 — Scaling curves\n"
     "val_loss vs #parameters for all paradigms"),
]

for fname, description in figures:
    path = os.path.join(FIG_DIR, fname)
    if not os.path.exists(path):
        print(f"[missing] {fname}")
        continue
    fig, ax = plt.subplots(figsize=(14, 6))
    ax.imshow(mpimg.imread(path))
    ax.axis("off")
    ax.set_title(description, fontsize=12, loc="left", pad=10)
    plt.tight_layout()
    plt.show()

### 9b — Training curves from pre-trained checkpoints

Every run saves `training_log.csv` with step, train loss, and val loss at every checkpoint.
Here we plot convergence for `diff_mha_768d_16b_Dense` and `elf_mha_768d_16b_Dense`
and read model metadata from `model_summary.json`.

In [ ]:
import json

import matplotlib.pyplot as plt
import pandas as pd

runs_to_plot = [
    ("Discrete Diffusion \u00b7 MHA", "runs/diff_mha_768d_16b_Dense",  "#4C72B0"),
    ("ELF \u00b7 MHA",               "runs/elf_mha_768d_16b_Dense",   "#DD8452"),
]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for idx, (label, run_dir, color) in enumerate(runs_to_plot):
    log_path  = os.path.join(run_dir, "training_log.csv")
    meta_path = os.path.join(run_dir, "model_summary.json")

    if not os.path.exists(log_path):
        print(f"[skip] {label} — {log_path} not found")
        continue

    df   = pd.read_csv(log_path)
    meta = json.load(open(meta_path)) if os.path.exists(meta_path) else {}

    title = label
    if meta:
        title += f"  ({meta.get('total_params_M', '?')}M params)"

    ax = axes[idx]
    ax.plot(df["step"], df["train_loss"], label="train", color=color)
    ax.plot(df["step"], df["val_loss"],   label="val",   color=color, linestyle="--", alpha=0.7)
    ax.set_title(title)
    ax.set_xlabel("step")
    ax.set_ylabel("loss")
    ax.legend()
    ax.grid(True, alpha=0.3)

    print(f"\n{label}")
    for k, v in meta.items():
        print(f"  {k}: {v}")
    print(f"  last step: {df['step'].iloc[-1]}  |  val_loss: {df['val_loss'].iloc[-1]:.4f}")

plt.tight_layout()
plt.show()

### 9c — Summary of available pre-trained runs

In [ ]:
import json

RUN_ROOT = "runs"
if os.path.isdir(RUN_ROOT):
    runs = sorted(os.listdir(RUN_ROOT))
    print(f"{'Run':<40}  {'Params (M)':>10}  {'Val loss':>10}  {'dtype':>8}")
    print("-" * 75)
    for r in runs:
        run_dir  = os.path.join(RUN_ROOT, r)
        meta_p   = os.path.join(run_dir, "model_summary.json")
        log_p    = os.path.join(run_dir, "training_log.csv")
        if not os.path.isdir(run_dir):
            continue
        meta     = json.load(open(meta_p))  if os.path.exists(meta_p) else {}
        last_val = ""
        if os.path.exists(log_p):
            df = pd.read_csv(log_p)
            last_val = f"{df['val_loss'].iloc[-1]:.4f}"
        n_m   = meta.get("total_params_M", "?")
        dtype = meta.get("dtype", "?")
        print(f"  {r:<38}  {str(n_m):>10}  {last_val:>10}  {dtype:>8}")
else:
    print("[skip] 'runs/' directory not found")

---
## CLI — same control from the terminal

Everything shown above is also available via CLI:

```bash
# Profile parameters, FLOPs and memory of a checkpoint
dantinox profile --run_dir runs/diff_mha_768d_16b_Dense/

# Generate text (discrete diffusion, 50 steps, streaming)
dantinox generate \
    --run_dir runs/diff_mha_768d_16b_Dense/ \
    --prompt  "Language models will change" \
    --n_steps 50 --stream

# Generate with ELF (flow-matching, CFG guidance scale)
dantinox generate \
    --run_dir runs/elf_mha_768d_16b_Dense/ \
    --prompt  "Language models will change" \
    --cfg_scale 1.5 --n_steps 30 --stream

# Train on WikiText-103 with 4 GPUs (2DP × 2TP)
dantinox train \
    --paradigm discrete --attention gqa --kv_heads 2 \
    --ffn moe --n_experts 8 --top_k 2 \
    --dim 768 --n_heads 12 --num_blocks 16 \
    --tp_size 2 --n_devices 4 \
    --dataset wikitext --lr 3e-4 --epochs 10

# Full benchmark suite (requires GPUs + dataset)
dantinox infbench --groups paradigm attention --n-trials 3
```

---

**DantinoX** · MIT license · [github.com/winstonsmith1897/dantinox](https://github.com/winstonsmith1897/dantinox)

```bash
pip install dantinox
```